In [ ]:
from flowmia import FlowMIA

## Treinamento e avaliação por modelo

Executa FlowMIA-GAN, DOMIAS e DCR para cada modelo gerativo, iterando sobre `MODELS` em vez de repetir manualmente o mesmo bloco de código por modelo (netshare/ctgan/tabula/TabDDPM/TabDDPM_synthcity).

In [ ]:
import os

import numpy as np

MODELS = [
    {
        "name": "netshare",
        "synth_path": "datasets/synthetic/netshare.csv",
        "numerical_cols": ["srcport", "dstport", "td", "pkt", "byt"],
        "num_epochs": 100,
    },
    {
        "name": "ctgan",
        "synth_path": "datasets/synthetic/ctgan.csv",
        "numerical_cols": ["srcport", "dstport", "pkt", "td", "byt"],
        "num_epochs": 100,
    },
    {
        "name": "tabula",
        "synth_path": "datasets/synthetic/tabula.csv",
        "numerical_cols": ["srcport", "dstport", "td", "pkt", "byt"],
        "num_epochs": 50,
    },
    {
        "name": "TabDDPM",
        "synth_path": "datasets/synthetic/TabDDPM.csv",
        "numerical_cols": ["srcport", "dstport", "td", "pkt", "byt"],
        "num_epochs": 50,
    },
    {
        "name": "TabDDPM_synthcity",
        "synth_path": "datasets/synthetic/TabDDPM_synthcity.csv",
        "numerical_cols": ["srcport", "dstport", "td", "pkt", "byt"],
        "num_epochs": 50,
    },
]

TEST_SIZE = 10000

flowmia_instances = {}
flowmiagan_results = {}

for model in MODELS:
    save_path = f"results_archive/resultados_oficiais/{model['name']}"
    config = {
        'member_path': 'datasets/real/cidds_train.csv',
        'non_member_path': 'datasets/reference/ton.csv',
        'synth_path': model['synth_path'],
        'test_path': 'datasets/real/cidds_test.csv',
        'categorical_cols': ['proto'],
        'numerical_cols': model['numerical_cols'],
        'ip_cols': ['srcip', 'dstip'],
        'label_col': 'label',
        'batch_size': 1000,
        'num_epochs': model['num_epochs'],
        'fcheckpoint': 100,
        'save_path': save_path,
        'use_wgan': True,
    }
    flowmia_instance = FlowMIA(config=config)
    flowmia_instances[model['name']] = flowmia_instance

    scores = flowmia_instance.flowmiagan(test_size=TEST_SIZE)
    flowmiagan_results[model['name']] = scores

    domias_scores, _ = flowmia_instance.domias(
        test_size=TEST_SIZE, save_path=f"{save_path}/domias", epochs=30, load=True,
    )
    os.makedirs(f"{save_path}/domias", exist_ok=True)
    np.save(f"{save_path}/domias/scores.npy", domias_scores)

    dcr_scores, _ = flowmia_instance.compute_dcr(test_size=TEST_SIZE)
    os.makedirs(f"{save_path}/dcr", exist_ok=True)
    np.save(f"{save_path}/dcr/scores.npy", dcr_scores)

## Plots Artigo

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

PLOT_MODELS = [("ctgan", "CTGAN"), ("netshare", "NetShare"), ("tabula", "Tabula")]
y_test = np.concatenate([np.ones(TEST_SIZE), np.zeros(TEST_SIZE)])

aucs = {"FlowMIA-GAN": [], "DOMIAS": [], "DCR": []}
roc_curves = {"FlowMIA-GAN": [], "DOMIAS": [], "DCR": []}

for name, _ in PLOT_MODELS:
    save_path = f"results_archive/resultados_oficiais/{name}"
    scores = flowmiagan_results[name]
    scores_flowmiagan = np.concatenate([scores['score_members'], scores['score_non_members']])
    scores_domias = np.load(f"{save_path}/domias/scores.npy")
    scores_dcr = np.load(f"{save_path}/dcr/scores.npy")

    for attack_name, s in [("FlowMIA-GAN", scores_flowmiagan), ("DOMIAS", scores_domias), ("DCR", scores_dcr)]:
        aucs[attack_name].append(roc_auc_score(y_test, s))
        roc_curves[attack_name].append(roc_curve(y_test, s))

colors = {
    "NetShare": "#1100ff",
    "CTGAN": "#ff0000",
    "Tabula": "#007230",
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

models = ['CTGAN', 'NetShare', 'Tabula']
attacks = list(aucs.keys())

# posição das barras
x = np.arange(len(attacks))
width = 0.25

plt.figure(figsize=(12, 12))

# plot das barras (uma por modelo)
for i, model in enumerate(models):
    values = [aucs[attack][i] for attack in attacks]
    plt.bar(x + i*width, values, width, label=model, color=colors[model])

# ajustes
plt.xticks(x + width, attacks)
plt.tick_params(axis='both', which='major', labelsize=25, width=3, length=10)
plt.ylabel('AUC', fontsize=30)
plt.xlabel('MIA Attacks', fontsize=30)


plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=3, fontsize=20)

os.makedirs('results_archive/root_plots', exist_ok=True)
plt.savefig('results_archive/root_plots/auc_scores.pdf')
plt.show()
	


In [ ]:
import matplotlib.pyplot as plt
import os

models = ['CTGAN', 'NetShare', 'Tabula']

os.makedirs('results_archive/root_plots', exist_ok=True)

for (attack_name, roc_list) in roc_curves.items():
	plt.figure(figsize=(7,6))
	for model_name, (fpr, tpr, _) in zip(models, roc_list):
		plt.plot(fpr, tpr, label=model_name, color=colors[model_name])

	# linha aleatória
	plt.plot([0, 1], [0, 1], linestyle='--')    

	plt.xlabel('FPR', fontsize=30)
	plt.ylabel('TPR', fontsize=30)
	plt.tick_params(axis='both', which='major', labelsize=20, width=3, length=10)
	plt.legend(fontsize=20)
	plt.tight_layout()
	plt.savefig(f'results_archive/root_plots/roc_curves_{attack_name}.pdf')
	plt.show()

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

classifiers = [
    				#MLPClassifier(hidden_layer_sizes=(128, 64, 64), random_state=42),
                  #  DecisionTreeClassifier(max_depth=12, min_samples_leaf=50),
                    #KNeighborsClassifier(n_neighbors=5),
                   RandomForestClassifier(n_estimators=100, random_state=42)
                ]


utility_dict = flowmia_instances['ctgan'].evaluate_utility(classifiers=classifiers, plot=True)

In [ ]:
utility_netshare = flowmia_instances['netshare'].evaluate_utility(classifiers=classifiers.copy(), plot=True)['RandomForestClassifier']['TSTR']
utility_ctgan = flowmia_instances['ctgan'].evaluate_utility(classifiers=classifiers.copy(), plot=True)['RandomForestClassifier']['TSTR']
utility_tabula = flowmia_instances['tabula'].evaluate_utility(classifiers=classifiers.copy(), plot=True)['RandomForestClassifier']['TSTR']

In [27]:
data = [utility_ctgan, utility_netshare, utility_tabula]

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]

os.makedirs('results_archive/root_plots', exist_ok=True)

# Generate one plot per metric
for metric in metrics:
    values = [data[i][metric] for i in range(3)]
    
    plt.figure()
    plt.bar(models, values, color=[colors[m] for m in models])
    
    plt.ylabel(metric, fontsize=35)
    plt.xlabel('Generative Model', fontsize=35)
    plt.ylim(0, 1.05)
    plt.tick_params(axis='both', which='major', labelsize=30, width=3, length=10)
    
    for i, v in enumerate(values):
        plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=30)
    
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'results_archive/root_plots/utility_{metric}.pdf')
    plt.show()

In [ ]:
# ctgan, netshare e tabula + net diffusion 